In [3]:
!pip install -q transformers accelerate datasets evaluate peft bitsandbytes sentence-transformers


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 33.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 41.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 108.2 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 87.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 2.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 30.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━

In [4]:
import os
import json
import time
import unicodedata
import torch
import numpy as np
import pandas as pd
from datasets import Dataset, DatasetDict
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model
import evaluate

In [5]:
HF_TOKEN = ""  
os.environ["HF_TOKEN"] = HF_TOKEN

In [6]:
class DatasetProcessor:
    def __init__(self, data_path):
        self.data_path = data_path
        self.dataset = None
        self.tokenizer = None

    def _normalize_text(self, text):
        if not isinstance(text, str):
            return ""
        return unicodedata.normalize("NFKC", text.strip().replace("\n", " "))

    def load_raw_dataset(self):
        df = pd.read_csv(self.data_path)
        df['Questions'] = df['Questions'].apply(self._normalize_text)
        df['Answers'] = df['Answers'].apply(self._normalize_text)
        df = df[(df['Questions'].str.len() > 10) & (df['Answers'].str.len() > 10)]
        df = df.drop_duplicates(subset=['Answers']).reset_index(drop=True)
        self.dataset = Dataset.from_pandas(df)
        print(f"Loaded {len(df)} examples after cleaning.")

    def prepare_instruction_dataset(self, tokenizer, max_seq_length=8192):
        self.tokenizer = tokenizer
        system_prompt = "আপনি একজন সহানুভূতিশীল কাউন্সেলর। বাংলায় সহানুভূতিশীলভাবে উত্তর দিন।"

        def format_example(example):
            text = f"<s>[INST] {system_prompt}\n\n{example['Questions']} [/INST] {example['Answers']}{tokenizer.eos_token}"
            tokenized = tokenizer(
                text,
                max_length=max_seq_length,
                truncation=True,
                padding="max_length"
            )
            return {"input_ids": tokenized["input_ids"], "attention_mask": tokenized["attention_mask"], "text": text}

        formatted = self.dataset.map(format_example, num_proc=4)
        train_test = formatted.train_test_split(test_size=0.2)
        val_test = train_test['test'].train_test_split(test_size=0.5)
        self.dataset = DatasetDict({
            'train': train_test['train'],
            'validation': val_test['train'],
            'test': val_test['test']
        })
        print(f"Train: {len(self.dataset['train'])}, Val: {len(self.dataset['validation'])}, Test: {len(self.dataset['test'])}")
        return self.dataset


In [7]:
class Evaluator:
    def __init__(self, model, tokenizer, dataset):
        self.model = model
        self.tokenizer = tokenizer
        self.dataset = dataset
        self.bleu = evaluate.load("sacrebleu")
        self.rouge = evaluate.load("rouge")

    def evaluate_model(self, max_samples=100):
        self.model.eval()
        test_subset = self.dataset['test'].select(range(min(max_samples, len(self.dataset['test']))))

        predictions = []
        references = []

        for ex in tqdm(test_subset, desc="Generating for evaluation"):
            inputs = self.tokenizer(ex['Questions'], return_tensors="pt").to("cuda")
            outputs = self.model.generate(**inputs, max_new_tokens=150, do_sample=True, temperature=0.7)
            response = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
            predictions.append(response)
            references.append([ex['Answers']])

        bleu_score = self.bleu.compute(predictions=predictions, references=references)['score']
        rouge_l = self.rouge.compute(predictions=predictions, references=[r[0] for r in references])['rougeL']

        print(f"BLEU: {bleu_score:.2f}, ROUGE-L: {rouge_l:.4f}")
        return {"bleu": bleu_score, "rouge_l": rouge_l}

In [8]:
class LLAMAFineTuner:
    def __init__(self, config):
        self.config = config
        self.model = None
        self.tokenizer = None
        self.dataset_processor = DatasetProcessor(config['data_path'])
        self.experiment_id = int(time.time())
        self.experiments_log = "LLAMAExperiments.jsonl"
        self.responses_log = "GeneratedResponses.jsonl"

    def build_tokenizer_and_model(self):
        print("Loading tokenizer and model...")
        model_name = self.config['model_name']

        self.tokenizer = AutoTokenizer.from_pretrained(model_name, token=HF_TOKEN)
        self.tokenizer.pad_token = self.tokenizer.eos_token

        quant_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True
        )

        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            device_map="auto",
            quantization_config=quant_config,
            offload_folder="./offload",
            llm_int8_enable_fp32_cpu_offload=True,
            token=HF_TOKEN
        )

    def apply_lora(self):
        print("Applying LoRA...")
        lora_config = LoraConfig(
            r=16,
            lora_alpha=32,
            target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
            lora_dropout=0.05,
            bias="none",
            task_type="CAUSAL_LM"
        )
        self.model = get_peft_model(self.model, lora_config)

    def train(self):
        print("Preparing dataset...")
        dataset = self.dataset_processor.prepare_instruction_dataset(self.tokenizer, max_seq_length=self.config['max_seq_length'])

        print("Starting training...")
        from transformers import Trainer

        training_args = TrainingArguments(
            output_dir=self.config['output_dir'],
            num_train_epochs=self.config['num_train_epochs'],
            per_device_train_batch_size=1,
            gradient_accumulation_steps=8,
            learning_rate=2e-4,
            fp16=True,
            logging_steps=20,
            save_steps=500,
            evaluation_strategy="steps",
            eval_steps=200,
            save_total_limit=2,
            gradient_checkpointing=True,
            optim="adamw_8bit",
            report_to="none"
        )

        trainer = Trainer(
            model=self.model,
            train_dataset=dataset['train'],
            eval_dataset=dataset['validation'],
            tokenizer=self.tokenizer,
            args=training_args,
            data_collator=lambda data: {
                "input_ids": torch.stack([torch.tensor(f["input_ids"]) for f in data]),
                "attention_mask": torch.stack([torch.tensor(f["attention_mask"]) for f in data]),
                "labels": torch.stack([torch.tensor(f["input_ids"]) for f in data])
            }
        )

        trainer.train()

        # Log experiment
        eval_metrics = trainer.evaluate()
        record = {
            "id": self.experiment_id,
            "model_name": self.config['model_name'],
            "lora_config": lora_config.to_dict(),
            "train_loss": trainer.state.log_history[-1].get('loss', 0.0),
            "val_loss": eval_metrics.get('eval_loss', 0.0),
            "metrics": eval_metrics,
            "timestamp": time.time()
        }
        with open(self.experiments_log, "a", encoding="utf-8") as f:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")

        return trainer

    def save_model(self):
        print("Saving model and tokenizer...")
        self.model.save_pretrained(self.config['output_dir'])
        self.tokenizer.save_pretrained(self.config['output_dir'])

In [9]:
config = {
    "data_path": "/kaggle/input/bengali-empathetic-conversations-corpus/BengaliEmpatheticConversationsCorpus .csv",
    "model_name": "meta-llama/Meta-Llama-3.1-8B-Instruct",
    "max_seq_length": 8192,
    "num_train_epochs": 1,
    "output_dir": "./llama-bengali-lora"
}


In [10]:
ft = LLAMAFineTuner(config)


In [18]:
# Force install compatible versions
!pip install -q --upgrade bitsandbytes==0.41.0
!pip install -q --upgrade transformers accelerate peft datasets evaluate sentence-transformers


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.6/92.6 MB 11.6 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 108.1 MB/s eta 0:00:0000:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.9/380.9 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 556.4/556.4 kB 33.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 512.3/512.3 kB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 493.7/493.7 kB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 89.0 MB/s eta 0:00:00:00:01


In [21]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

quant_config = BitsAndBytesConfig(
    load_in_8bit=True,  # <- 8-bit instead of 4-bit
    bnb_8bit_compute_dtype=torch.float16
)

model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Meta-Llama-3.1-8B-Instruct",
    device_map="auto",
    quantization_config=quant_config,
    offload_folder="./offload",
    llm_int8_enable_fp32_cpu_offload=True,
    token=HF_TOKEN
)


ImportError: Using `bitsandbytes` 8-bit quantization requires the latest version of bitsandbytes: `pip install -U bitsandbytes`

In [23]:
ft.build_tokenizer_and_model()


Loading tokenizer and model...


ImportError: Using `bitsandbytes` 4-bit quantization requires the latest version of bitsandbytes: `pip install -U bitsandbytes`

In [24]:
ft.apply_lora()


Applying LoRA...


AttributeError: 'NoneType' object has no attribute '__dict__'

In [25]:
trainer = ft.train()


Preparing dataset...


AttributeError: 'NoneType' object has no attribute 'map'

In [26]:
evaluator = Evaluator(ft.model, ft.tokenizer, ft.dataset_processor.dataset)
metrics = evaluator.evaluate_model()

ImportError: To be able to use evaluate-metric/sacrebleu, you need to install the following dependencies['sacrebleu'] using 'pip install sacrebleu' for instance'

In [ ]:
ft.save_model()

print("\nFinal Metrics:", metrics)